# Инференс: запрос → рекомендации

Пользователь (история прослушиваний + профиль) пишет запрос на русском, в ответ — текст и список треков.
Вся логика в пакете `recsys/`, здесь только запуск.

* `main.ipynb` — эксперименты: метрики, сравнение вариантов, разбор шагов, LLM
* `examples/candgen.ipynb` — проверка сервисов BM25 и HNSW руками

## 1. Настройки

* `USE_CANDGEN = True` — BM25 с сервера по HTTP (`configs/candgen.yaml` + `configs/server.yaml`). HNSW на сервере
  пока нет, поэтому поиск по эмбеддингам считается здесь. Адрес и ключ — `BM25_URL`, `BM25_API_KEY`
  (файл `.env` или секреты проекта DataSphere).
* `DATA = "crs"` — настоящий датасет из `DATA_DIR` (файлы целиком или частями `train-00000-of-00078.parquet`);
  `"synthetic"` — сгенерированные данные, работает без файлов.

In [2]:
USE_CANDGEN = True
DATA = "synthetic"         # "crs" — настоящий датасет
DATA_DIR = "data/full"     # tracks_meta, train, test_public
N_USERS = 300              # сколько пользователей загрузить (None — все, долго)

# адреса и ключи — из файла .env или секретов DataSphere; можно и здесь:
# import os; os.environ["BM25_URL"] = "http://<IP>:8000"; os.environ["BM25_API_KEY"] = "..."

In [ ]:
%pip install -q -r requirements.txt

## 2. Каталог и пайплайн

Загружается один раз. Для сервисов печатается, доступны ли они и какая версия индекса.

In [3]:
from dataclasses import replace

from recsys.config import CANDGEN_CONFIGS, load_config
from recsys.data import load_data
from recsys.pipeline import Pipeline
from recsys.schemas import Message

files = ["configs/default.yaml"] + (CANDGEN_CONFIGS if USE_CANDGEN else [])
cfg = load_config(files, {"data": {"source": DATA, "crs": {"dir": DATA_DIR, "n_users": N_USERS}}})
data = load_data(cfg)
pipe = Pipeline.from_config(cfg, data.catalog)

for source in pipe.retrievers:
    if getattr(source, "remote", False):
        print(source.name, source.health())

[data] synthetic: DataBundle(tracks=5000, requests: train=1032, test_public=1032) за 1.9 c
bm25_genres {'url': 'http://localhost:8001', 'status': 'ok', 'index_version': '0f9169788cea', 'n_items': 5}
bm25_tags {'url': 'http://localhost:8001', 'status': 'ok', 'index_version': 'ae525fdfcf56', 'n_items': 5}
bm25_title {'url': 'http://localhost:8001', 'status': 'ok', 'index_version': '5a5da1de58c8', 'n_items': 5}
hnsw_audio {'url': 'http://localhost:8002', 'status': 'ok', 'index_version': '8af6209d5fcd', 'n_items': 5}


## 3. Запрос

Берём пользователя из датасета: его история и профиль. `TEXT` — свой запрос; `None` — запрос из датасета
(тогда известен трек, который пользователь на самом деле послушал).

In [4]:
user = data.requests[0]
TEXT = "хочу что-нибудь спокойное под вечер, без рэпа"     # None — запрос из датасета

request = replace(user, dialog=[Message("user", TEXT)]) if TEXT else user
print("запрос:", request.dialog[-1].text)
print("треков в истории:", len(request.history))
print("профиль:", request.user_info[:200], "...")

запрос: хочу что-нибудь спокойное под вечер, без рэпа
треков в истории: 38
профиль: Слушает музыку с 2010 года; до периода 332 прослушиваний, 38 разных треков, 28 артистов. Любимые артисты: The Static Hearts, The Hollow Tides, Wild Ghosts, The Electric Waves, Crystal Foxes, Static Ow ...


## 4. Ответ

In [5]:
response = pipe.run(request, debug=True)
print(response.text)
response.to_frame()

Привет! Вот что я подобрал (calm, chill):
1. The Static Hearts — Night [jazz, night, instrumental] — вы уже слушали этот трек
2. The Hollow Tides — Glass Mirror River [jazz, instrumental, night] — вы уже слушали этот трек
3. The Static Hearts — Blue [jazz, saxophone, instrumental] — вы уже слушали этот трек
4. The Hollow Tides — Mountain Garden Summer [jazz, instrumental, saxophone] — вы уже слушали этот трек
5. Static Foxes — Train Paper [ambient, calm, atmospheric] — совпадает: calm, chill; вы уже слушали этот трек
6. The Static Hearts — Road Window Rain [jazz, smooth jazz, night] — вы уже слушали этот трек
7. Wild Ghosts — Mountain Morning [jazz, saxophone, relaxing] — вы уже слушали этот трек
8. Static Foxes — Garden Rain [ambient, chill, relaxing] — совпадает: calm, chill; вы уже слушали этот трек
9. Crystal Foxes — Light [jazz, instrumental, smooth jazz] — вы уже слушали этот трек
10. The Static Saints — Road Glass Heart [ambient, chill, relaxing] — совпадает: chill; вы уже слуша

,rank,track_id,artist,title,score,tags,reasons
0,1,T000000000002352,The Static Hearts,Night,0.0225,"jazz, night, instrumental",вы уже слушали этот трек
1,2,T000000000000915,The Hollow Tides,Glass Mirror River,0.0214,"jazz, instrumental, night, saxophone, sad, mal...",вы уже слушали этот трек
2,3,T000000000004585,The Static Hearts,Blue,0.0209,"jazz, saxophone, instrumental, smooth jazz, ni...",вы уже слушали этот трек
3,4,T000000000000876,The Hollow Tides,Mountain Garden Summer,0.0198,"jazz, instrumental, saxophone, relaxing, night",вы уже слушали этот трек
4,5,T000000000001079,Static Foxes,Train Paper,0.0189,"ambient, calm, atmospheric, chill, instrumenta...","совпадает: calm, chill; вы уже слушали этот трек"
5,6,T000000000004464,The Static Hearts,Road Window Rain,0.0184,"jazz, smooth jazz, night",вы уже слушали этот трек
6,7,T000000000001519,Wild Ghosts,Mountain Morning,0.0181,"jazz, saxophone, relaxing",вы уже слушали этот трек
7,8,T000000000002790,Static Foxes,Garden Rain,0.0180,"ambient, chill, relaxing, calm, summer","совпадает: calm, chill; вы уже слушали этот трек"
8,9,T000000000003394,Crystal Foxes,Light,0.0179,"jazz, instrumental, smooth jazz, night, saxoph...",вы уже слушали этот трек
9,10,T000000000002375,The Static Saints,Road Glass Heart,0.0175,"ambient, chill, relaxing, atmospheric",совпадает: chill; вы уже слушали этот трек


Если запрос из датасета: на каком месте трек, который пользователь послушал.

In [6]:
if not TEXT:
    target = request.target_ids[0]
    place = response.track_ids.index(target) + 1 if target in response.track_ids else "нет в выдаче"
    print("цель:", request.meta["artist"], "—", request.meta["title"], "| место:", place)

## 5. Почему такая выдача

Как понят запрос (английские теги — именно они уходят в BM25) и сколько кандидатов дал каждый источник.
`relisten` — треки из своей истории, `bm25_*` и `hnsw_audio` — сервисы, остальные считаются здесь.

In [7]:
print("понял как:", response.summary.text)
print("теги:", response.summary.include_tags, "| исключить:", response.summary.exclude_tags)
print("кандидатов:", response.debug["n_candidates"])

понял как: User wants calm, chill; mood: calm; energy: low; avoid rap, hip hop; generally likes jazz, ambient, punk, metal, soul.
теги: ['calm', 'chill'] | исключить: ['rap', 'hip hop']
кандидатов: {'relisten': 38, 'title': 0, 'history': 200, 'popular': 100, 'bm25_genres': 0, 'bm25_tags': 0, 'bm25_title': 0, 'hnsw_audio': 0}


## 6. Следующая реплика

Диалог продолжается: уже показанное не повторяется, новые пожелания перекрывают старые.

In [8]:
request2 = request.next_turn(response, "а теперь поэнергичнее")
response2 = pipe.run(request2)
print(response2.text)
response2.to_frame()

Привет! Вот что я подобрал (calm, chill):
1. The Hollow Tides — Heart [jazz, saxophone, instrumental] — вы уже слушали этот трек
2. Electric Comets — Rain Blue Echo [soul, female vocalists, rnb] — вы уже слушали этот трек
3. Static Foxes — Night Heart [ambient, relaxing, atmospheric] — совпадает: calm, chill; вы уже слушали этот трек
4. Violet Mirrors — Garden [classical, piano, orchestral] — совпадает: calm; вы уже слушали этот трек
5. Cosmic Rivers — Mirror River [jazz, smooth jazz, instrumental] — вы уже слушали этот трек
6. Static Owls — Night [ambient, chill, calm] — совпадает: calm, chill; вы уже слушали этот трек
7. Wild Ghosts — Storm Night Mountain [jazz, night, relaxing] — вы уже слушали этот трек
8. Northern Bridges — Shadows Ocean [ambient, chill, atmospheric] — совпадает: calm, chill; вы уже слушали этот трек
9. Paper Engines — Silence [jazz, night, relaxing] — вы уже слушали этот трек
10. Gentle Tides — Storm Stars Night [ambient, atmospheric, instrumental] — совпадает: c

,rank,track_id,artist,title,score,tags,reasons
0,1,T000000000000912,The Hollow Tides,Heart,0.0190,"jazz, saxophone, instrumental, relaxing, night",вы уже слушали этот трек
1,2,T000000000002092,Electric Comets,Rain Blue Echo,0.0171,"soul, female vocalists, rnb, night",вы уже слушали этот трек
2,3,T000000000001197,Static Foxes,Night Heart,0.0171,"ambient, relaxing, atmospheric, chill, calm","совпадает: calm, chill; вы уже слушали этот трек"
3,4,T000000000000532,Violet Mirrors,Garden,0.0168,"classical, piano, orchestral, instrumental, ca...",совпадает: calm; вы уже слушали этот трек
4,5,T000000000003038,Cosmic Rivers,Mirror River,0.0165,"jazz, smooth jazz, instrumental, relaxing, nig...",вы уже слушали этот трек
5,6,T000000000000093,Static Owls,Night,0.0164,"ambient, chill, calm, instrumental, relaxing, ...","совпадает: calm, chill; вы уже слушали этот трек"
6,7,T000000000000425,Wild Ghosts,Storm Night Mountain,0.0164,"jazz, night, relaxing, instrumental, male voca...",вы уже слушали этот трек
7,8,T000000000000279,Northern Bridges,Shadows Ocean,0.0161,"ambient, chill, atmospheric, calm, instrumental","совпадает: calm, chill; вы уже слушали этот трек"
8,9,T000000000004980,Paper Engines,Silence,0.0159,"jazz, night, relaxing, saxophone, instrumental...",вы уже слушали этот трек
9,10,T000000000004385,Gentle Tides,Storm Stars Night,0.0159,"ambient, atmospheric, instrumental, calm, chil...","совпадает: calm, chill; вы уже слушали этот трек"
